In [0]:
import dlt
from pyspark.sql.functions import col, to_timestamp, regexp_replace, lit

# --------- CONFIG ---------
CATALOG = spark.conf.get("pipeline.catalog", "dev")
SCHEMA = spark.conf.get("pipeline.schema", "retail")
RAW_ORDERS_PATH = spark.conf.get("pipeline.raw_orders_path", "/Volumes/dev/retail/raw/orders")
# --------------------------

@dlt.view(
    name="bronze_orders_src",
    comment="Raw incremental orders from Volume or cloud path"
)
def bronze_orders_src():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load(RAW_ORDERS_PATH)
    )

@dlt.table(
    name="bronze_orders",
    comment="Bronze: raw ingested orders as-is"
)
def bronze_orders():
    return dlt.read_stream("bronze_orders_src")

@dlt.table(
    name="silver_orders",
    comment="Silver: cleaned & conformed orders"
)
@dlt.expect_or_drop("non_negative_amount", "amount >= 0")
@dlt.expect_or_drop("mandatory_keys", "order_id IS NOT NULL AND customer_id IS NOT NULL")
def silver_orders():
    df = dlt.read_stream("bronze_orders")
    return (
        df.dropDuplicates(["order_id"])
        .withColumn("order_ts", to_timestamp(col("order_ts")))
        .withColumn("currency", regexp_replace(col("currency"), r"[^A-Z]", ""))
        .withColumn("ingest_catalog", lit(CATALOG))
        .withColumn("ingest_schema", lit(SCHEMA))
    )

@dlt.table(
    name="gold_sales_daily",
    comment="Gold: daily sales by product and region"
)
def gold_sales_daily():
    s = dlt.read_stream("silver_orders")
    return (
        s.groupBy("order_date", "product_id", "region")
        .agg({"amount": "sum", "order_id": "count"})
        .withColumnRenamed("sum(amount)", "total_amount")
        .withColumnRenamed("count(order_id)", "order_count")
    )